# IMDb Sentiment Analysis - Model Training

This notebook trains a DistilBERT model on the IMDb dataset.

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path('../')))

import torch
from src.config import config
from src.data import load_imdb_from_csv, IMDbDataset, create_data_loaders
from src.model import get_model, save_model
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f'PyTorch version: {torch.__version__}')
print(f'Device: {config.device}')

PyTorch version: 2.12.1+cpu
Device: cpu


In [4]:
print('Loading IMDb dataset from CSV...')
csv_path = config.data_dir / 'IMDB Dataset.csv'
train_texts, train_labels, test_texts, test_labels = load_imdb_from_csv(csv_path)

if config.train_sample_size:
    train_texts = train_texts[:config.train_sample_size]
    train_labels = train_labels[:config.train_sample_size]
if config.val_sample_size:
    test_texts = test_texts[:config.val_sample_size]
    test_labels = test_labels[:config.val_sample_size]

print(f'Training on {len(train_texts)} samples')
print(f'Validating on {len(test_texts)} samples')

Loading IMDb dataset from CSV...
Loading dataset from data\IMDB Dataset.csv...
Loaded 50000 reviews
Columns: ['review', 'sentiment']
Train samples: 40000
Test samples: 10000
Positive ratio (train): 50.00%
Training on 1000 samples
Validating on 200 samples


In [5]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

train_dataset = IMDbDataset(train_texts, train_labels, tokenizer, config.max_length)
test_dataset = IMDbDataset(test_texts, test_labels, tokenizer, config.max_length)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches: {len(test_loader)}')

Train batches: 125
Test batches: 25


In [ ]:
print('Initializing DistilBERT model...')
model = get_model(config)
model.to(config.device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

Initializing DistilBERT model...


In [ ]:
optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

total_steps = len(train_loader) * config.num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=config.warmup_steps,
    num_training_steps=total_steps
)

print('Optimizer and scheduler initialized!')

In [ ]:
def train_epoch(model, data_loader, optimizer, device, scheduler=None):
    model.train()
    losses = []
    correct = 0
    total = 0
    
    for batch in tqdm(data_loader, desc='Training'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask, labels)
        loss = outputs.loss
        logits = outputs.logits
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        
        losses.append(loss.item())
        _, preds = torch.max(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return np.mean(losses), correct / total

def eval_model(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    losses = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask, labels)
            loss = outputs.loss
            logits = outputs.logits
            
            losses.append(loss.item())
            _, preds = torch.max(logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            actual_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(actual_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(actual_labels, predictions, average='binary')
    
    return {'loss': np.mean(losses), 'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

print('Training functions defined!')

In [ ]:
print('Starting training...')
print('='*60)

train_losses = []
val_accuracies = []
best_accuracy = 0

for epoch in range(config.num_epochs):
    print(f'\nEpoch {epoch+1}/{config.num_epochs}')
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, config.device, scheduler)
    train_losses.append(train_loss)
    
    val_metrics = eval_model(model, test_loader, config.device)
    val_accuracies.append(val_metrics['accuracy'])
    
    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'Val Loss: {val_metrics["loss"]:.4f} | Val Acc: {val_metrics["accuracy"]:.4f}')
    print(f'Precision: {val_metrics["precision"]:.4f} | Recall: {val_metrics["recall"]:.4f} | F1: {val_metrics["f1"]:.4f}')
    
    if val_metrics['accuracy'] > best_accuracy:
        best_accuracy = val_metrics['accuracy']
        save_model(model, config, tokenizer)
        print(f'✓ New best model saved! Accuracy: {best_accuracy:.4f}')

print('\n' + '='*60)
print(f'Training complete! Best validation accuracy: {best_accuracy:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, config.num_epochs+1), train_losses, marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Over Epochs')
axes[0].grid(True)

axes[1].plot(range(1, config.num_epochs+1), val_accuracies, marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy Over Epochs')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from src.predict import predict_sentiment

test_reviews = [
    'This movie was absolutely fantastic! The acting was great and the plot kept me engaged throughout.',
    'Terrible film, complete waste of time. The acting was poor and the story made no sense.',
    'It was an okay movie, nothing special but not terrible either.',
    'One of the best films I have ever seen! Brilliant performances and amazing direction.',
    'I walked out of the theater after 30 minutes. Absolutely dreadful.'
]

print('Testing model with sample reviews:')
print('='*80)

for review in test_reviews:
    sentiment, confidence = predict_sentiment(review)
    print(f'\nReview: {review}')
    print(f'Prediction: {sentiment.upper()} (confidence: {confidence:.2%})')